In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mesa_reader
from plotly.subplots import make_subplots
import math
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as pyo

pio.templates.default = "plotly"

It should be noted that `core_mass` is calculated as the mass of the outermost `burning` zone AKA the first zone fully within the core.

Core compositions calculations are a bit more complicated. We need to sum over the core zones and produce a mass fraction for each element.

It goes something like this:

$$\frac{1}{m_*}\sum_{i=n_i}^{n_f} (m_i \times x_i)$$

Where:

$
\begin{align}
m_* &= \text{The mass of the star in kg} \\
n_i &= \text{The first zone in the core} \\
n_f &= \text{The last zone in the core} \\
m_i &= \text{The mass of the zone in kg} \\
x_i &= \text{The mass fraction of the element in the zone}
\end{align}
$

This is repeated for each element.

In [2]:
SOLAR_MASS = 1.989 * 10 ** 30  # kg
SOLAR_RADIUS = 696340 * 1000  # m

elements = {
    'h1': 'H',
    'he3': 'He3',
    'he4': 'He',
    'c12': 'C',
    'n14': 'N',
    'o16': 'O',
    'ne20': 'Ne',
    'mg24': 'Mg',
}


HE3_MASS = 3.016029
ATOMIC_UNIT = 1.6605402e-27  # kg

## Data object for stellar profiles, including detailed temperature-density profiles, core radius/mass, reaction rates, composition, etc.
class Zone:
    def __init__(self, zone_number, thickness, top, temperature, mass, next_zone_mass, reaction_rates,
                 composition: dict[str, float]):
        """
        :param zone_number: Number of the zone
        :param thickness: Thickness of the zone in log10 cm
        :param top: Radius at outer boundary of the zone in log10 Rsun
        :param temperature: Temperature at the center of the zone in log10 K
        :param mass: Mass from the outer edge of the zone to the center in log10 Msun
        :param next_zone_mass: Mass from the outer edge of the next zone inwards to the center of the star in log10 Msun
        :param reaction_rates: Raw reactions/second of he3+he3->h1+h1+he4 in the zone
        :param composition: Mass fractions of the zone
        """

        self.zone_number = zone_number
        self.thickness = (10 ** thickness) / 100  # log10 cm => m
        self.top = (10 ** top) * SOLAR_RADIUS  # log10 Rsun => m
        bottom = self.top - self.thickness  # m
        self.temperature = 10 ** temperature  # log10 K => K
        self.mass = (mass - next_zone_mass) * SOLAR_MASS  # Msun => kg
        self.mass_coordinate = mass * SOLAR_MASS  # Msun => kg
        self.reaction_rates = reaction_rates
        self.composition = composition

        self.volume = 4 / 3 * np.pi * (np.power(self.top, 3) - np.power(bottom, 3))  # Volume of the shell, m^3
        self.density = self.mass / self.volume  # kg/m^3

        self.he3_number_density = (self.composition['He3'] * self.density) / (HE3_MASS * ATOMIC_UNIT)

    def to_record(self):
        record = {
            'zone_number': self.zone_number,
            'thickness': self.thickness,
            'top': self.top,
            'temperature': self.temperature,
            'mass': self.mass,
            'reaction_rates': self.reaction_rates,
            'volume': self.volume,
            'density': self.density,
            'he3_number_density': self.he3_number_density,
        }

        for element in self.composition:
            record[element] = self.composition[element]

        return record

    def as_profile_record(self):
        return {
            'core_temperature': self.temperature,
            'core_density': self.density,
            'reaction_rates_per_unit_volume': self.reaction_rates / self.volume,
        }


class StellarProfile:
    def __init__(self, source_id, batch_idx, iteration_idx, mass, core_radius, radius, original_radius, age,
                 original_age, zones: list[Zone]):
        """
        :param source_id: Source ID of the star
        :param batch_idx: Batch index of the simulation
        :param iteration_idx: Iteration index of the simulation
        :param mass: Mass of the star in Msun
        :param core_radius: Radius of the core in log10 Rsun
        :param core_radius: Size of core relative to the simulation radius (which is in log10 Rsun)
        :param radius: Radius of the simulated star in log10 Rsun
        :param original_radius: Original radius of the star for error calculations Rsun
        :param age: Age of the simulated star in years
        :param original_age: Original age of the star for error calculations
        :param zones: All profile zones, will be filtered to only include the core
        """

        self.source_id = source_id
        self.batch_idx = batch_idx
        self.iteration_idx = iteration_idx
        self.mass = mass * SOLAR_MASS  # Msun => kg
        self.radius = (10 ** radius) * SOLAR_RADIUS  # log10 Rsun => m
        self.normalized_core_radius = core_radius
        self.core_radius = core_radius * self.radius  # Normalized (Rsun) => m
        self.original_radius = original_radius * SOLAR_RADIUS  # Rsun => m
        self.age = age
        self.original_age = original_age

        burning_zones = []
        records = []

        for zone in zones:
            if zone.top < self.core_radius:
                burning_zones.append(zone)
                records.append(zone.to_record())

        ### Sort by zone numbers
        burning_zones.sort(key=lambda x: x.zone_number)

        self.burning_zones = pd.DataFrame(records)

        ### Calculate core mass
        self.core_mass = self.burning_zones['mass'].sum()

        self.core_volume = 4 / 3 * np.pi * np.power(self.core_radius, 3)  # m^3
        self.core_density = (self.burning_zones['density'] * self.burning_zones['mass']).sum() / self.core_mass
        self.core_temperature = (self.burning_zones['temperature'] * self.burning_zones['mass']).sum() / self.core_mass

        self.reaction_rates = self.burning_zones['reaction_rates'].sum()
        self.reaction_rates_per_unit_volume = self.reaction_rates / self.core_volume  # reactions/second/m^3

        self.he3_number_density = (self.burning_zones['he3_number_density'] * self.burning_zones['mass']).sum() / self.core_mass

        # Core composition
        self.composition = {}

        for element in elements.values():
            self.composition[element] = 0
            mass_sum = (self.burning_zones[element] * self.burning_zones['mass']).sum()
            self.composition[element] = mass_sum / self.core_mass

    @staticmethod
    def error(original, calculated):
        return (calculated - original) / original

    def to_record(self):
        original_core_radius = self.normalized_core_radius * self.original_radius
        original_volume = 4 / 3 * np.pi * np.power(original_core_radius, 3)
        original_reaction_rates_per_unit_volume = self.reaction_rates / original_volume

        record = {
            'source_id': self.source_id,
            'batch_idx': self.batch_idx,
            'iteration_idx': self.iteration_idx,
            'mass': self.mass,
            'core_radius': self.core_radius,
            'core_radius_error': StellarProfile.error(original_core_radius, self.core_radius),
            'original_core_radius': original_core_radius,
            'radius': self.radius,
            'radius_error': StellarProfile.error(self.original_radius, self.radius),
            'original_radius': self.original_radius,
            'age': self.age,
            'original_age': self.original_age,
            'age_error': StellarProfile.error(self.original_age, self.age),
            'core_mass': self.core_mass,
            'core_volume': self.core_volume,
            'core_volume_error': StellarProfile.error(original_volume, self.core_volume),
            'original_core_volume': original_volume,
            'core_density': self.core_density,
            'core_temperature': self.core_temperature,
            'reaction_rates_per_unit_volume': self.reaction_rates_per_unit_volume,
            'reaction_rates_error': StellarProfile.error(original_reaction_rates_per_unit_volume,
                                                         self.reaction_rates_per_unit_volume),
            'original_reaction_rates_per_unit_volume': original_reaction_rates_per_unit_volume,
            'burning_zones': self.burning_zones.to_dict(orient='records'),
            'he3_number_density': self.he3_number_density,
        }

        for element in self.composition:
            record[element] = self.composition[element]

        return record

    def temperature_density_profile(self):
        densities = self.burning_zones['density']
        temperatures = self.burning_zones['temperature']
        radii = self.burning_zones['top']

        return [radii, temperatures, densities]

    def temperature_density_profile_2d(self):
        densities = self.burning_zones['density']
        temperatures = self.burning_zones['temperature']

        return [densities, temperatures]

    def temperature_profile(self):
        temperatures = self.burning_zones['temperature']
        radii = self.burning_zones['top']

        return [radii, temperatures]

    def density_profile(self):
        densities = self.burning_zones['density']
        radii = self.burning_zones['top']

        return [radii, densities]

## Parsing simulation results

In [16]:
output_headers = ['source_id', 'batch_idx', 'iteration_idx', 'radius', 'e_radius', 'mass', 'e_mass', 'age', 'y', 'z',
                  'log_cntr_P', 'log_cntr_Rho', 'log_cntr_T', 'core_radius', 'core_mass',
                  'H', 'He3', 'He', 'C', 'N', 'O', 'Ne', 'Mg', 'pp_rate']
output = []

profiles = {}  # source_id -> StellarProfile

records = []

all_zones = []

def load(name):
    data = mesa_reader.MesaLogDir(f'simulate/Tests/{name}', memoize_profiles=False)

    history_data: mesa_reader.MesaData = data.history_data
    last_profile: mesa_reader.MesaData = data.profile_data()

    row_data = []
    headers = []

    for i in history_data.bulk_data[-1]:
        row_data.append(i)

    for i in history_data.bulk_names:
        headers.append(i)

    df = pd.DataFrame([row_data], columns=headers)
    core_radius = 0

    radius = df['log_R'].iloc[-1]
    original_radius = 1

    age = df['star_age'].iloc[-1]
    original_age = 4.603e9

    for i in range(1, 10):
        burn_type = int(df[f'burn_relr_type_{i}'].iloc[-1])
        val = float(df[f'burn_relr_top_{i}'].iloc[-1])
        if burn_type != -9999 and val != 1 and val > 0:
            core_radius = val
        elif burn_type == -9999 and val == 1:
            break

    profile_data_raw = pd.DataFrame(last_profile.bulk_data, columns=last_profile.bulk_names)
    profile_data_raw['next_mass'] = profile_data_raw['mass'].shift(-1)
    profile_data_raw['next_mass'].fillna(0, inplace=True)

    zones = []

    log_dr_array = profile_data_raw['log_dr'].values
    logR_array = profile_data_raw['logR'].values
    logT_array = profile_data_raw['logT'].values
    mass_array = profile_data_raw['mass'].values
    next_mass_array = profile_data_raw['next_mass'].values
    raw_rate_array = profile_data_raw['raw_rate_r_he3_he3_to_h1_h1_he4'].values

    element_names = list(elements.keys())
    element_columns = {el: profile_data_raw[el].values for el in element_names}
    element_mapped_names = [elements[el] for el in element_names]

    for i in range(len(profile_data_raw)):
        composition = {element_mapped_names[j]: element_columns[element_names[j]][i] for j in
                       range(len(element_names))}
        top = logR_array[i]
        zone = Zone(
            zone_number=profile_data_raw.index[i],
            thickness=log_dr_array[i],
            top=top,
            temperature=logT_array[i],
            mass=mass_array[i],
            next_zone_mass=next_mass_array[i],
            reaction_rates=raw_rate_array[i],
            composition=composition,
        )

        # r = (10 ** radius) * SOLAR_RADIUS * core_radius
        #
        # if top < r:
        #     all_zones.append(zone.as_profile_record())

        # Append a new Zone
        zones.append(zone)

    profile = StellarProfile(name, 0, 0, 1, core_radius, radius,
                             original_radius, age, original_age, zones)

    records.append(profile.to_record())
    profiles[name] = profile

load('actual')
load('predicted')

df = pd.DataFrame(records)
df.set_index('source_id', inplace=True)

# zone_df = pd.DataFrame(all_zones)
df

,batch_idx,iteration_idx,mass,core_radius,core_radius_error,original_core_radius,radius,radius_error,original_radius,age,...,burning_zones,he3_number_density,H,He3,He,C,N,O,Ne,Mg
source_id,,,,,,,,,,,,,,,,,,,,,
actual,0,0,1.989000e+30,1.463230e+08,0.111432,1.316527e+08,7.739344e+08,0.111432,696340000,4.603000e+09,...,"[{'zone_number': 687, 'thickness': 1385338.957...",1.461500e+27,0.579804,0.000157,0.407581,0.000501,0.002547,0.005652,0.001272,0.002487
predicted,0,0,1.989000e+30,1.464721e+08,0.117141,1.311134e+08,7.779100e+08,0.117141,696340000,4.603000e+09,...,"[{'zone_number': 684, 'thickness': 1384004.472...",1.447379e+27,0.579373,0.000154,0.408568,0.000470,0.002447,0.005396,0.001215,0.002376


In [21]:
sum = 0.002078457804248116 + 0.005155311164153693 + 0.0005949738414880656 + 0.0009841850204788797 + 0.0005805741173052164 + 4.458910579102157e-05 + 0.0006406844819846909 + 0.00031202780025242665 + 3.3137513659570625e-06 + 5.7685689246417394e-05 + 1.600047278538962e-05 + 9.506819274159204e-06 + 0.0011095486271893715 + 7.026698401873529e-05

print(0.002078457804248116 / sum)
print(0.005155311164153693 / sum)
print(0.0005949738414880656 / sum)
print(0.0009841850204788797 / sum)
print(0.0005805741173052164 / sum)
print(4.458910579102157e-05 / sum)
print(0.0006406844819846909 / sum)
print(0.00031202780025242665 / sum)
print(3.3137513659570625e-06 / sum)
print(5.7685689246417394e-05 / sum)
print(1.600047278538962e-05 / sum)
print(9.506819274159204e-06 / sum)
print(0.0011095486271893715 / sum)
print(7.026698401873529e-05 / sum)

0.17829933908052542
0.44224548193586005
0.051039497886703145
0.08442776097050356
0.04980422560958677
0.0038250514763790303
0.05496075958989376
0.026767130151041794
0.0002842683056734314
0.004948534555774403
0.0013725915997813256
0.0008155371688932485
0.09518200778539983
0.006027813883984313


In [22]:
0.17829933908052542 + 0.44224548193586005 + 0.051039497886703145 + 0.08442776097050356 + 0.04980422560958677 + 0.0038250514763790303 + 0.05496075958989376 + 0.026767130151041794 + 0.0002842683056734314 + 0.004948534555774403 + 0.0013725915997813256 + 0.0008155371688932485 + 0.09518200778539983 + 0.006027813883984313

1.0